# RECARD: Multi-Model Fraud Detection & Behavioral Profiling

This notebook compiles the complete end-to-end Machine Learning pipeline discussed:
1. **K-Nearest Neighbors (KNN)**
2. **Random Forest (RF)**
3. **XGBoost (XGB)**
4. **Recurrent Neural Network (RNN - LSTM)** for sequential behavior.

In [ ]:
# 1. Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.metrics import precision_recall_curve, auc

import shap
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, SimpleRNN

import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

In [ ]:
# 2. Download Datasets
print('Downloading Datasets via kagglehub...')
sparkov_path = kagglehub.dataset_download('kartik2112/fraud-detection')
ulb_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
paysim_path = kagglehub.dataset_download('ealaxi/paysim1')
print('Downloads complete.')

In [ ]:
# 3. Chronological Splitting & Pipelining (Including KNN now)
csv_file = [f for f in os.listdir(ulb_path) if f.endswith('.csv')][0]
df = pd.read_csv(os.path.join(ulb_path, csv_file))
df = df.sort_values('Time')

X = df.drop('Class', axis=1)
y = df['Class']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Build Pipelines for KNN, RF, and XGB
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
])

rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=10, random_state=42, n_jobs=-1))
])

xgb_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', xgb.XGBClassifier(scale_pos_weight=577, max_depth=6, learning_rate=0.1, random_state=42))
])
print('Pipelines constructed.')

In [ ]:
# 4. Train Models
print("Training K-Nearest Neighbors...")
knn_pipeline.fit(X_train, y_train)

print("Training Random Forest...")
rf_pipeline.fit(X_train, y_train)

print("Training XGBoost...")
xgb_pipeline.fit(X_train, y_train)

knn_probs = knn_pipeline.predict_proba(X_test)[:, 1]
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]
xgb_probs = xgb_pipeline.predict_proba(X_test)[:, 1]
print('Training complete.')

In [ ]:
# 5. Multi-Model Scikit-Learn Visualizations (PR Curves)
knn_prec, knn_rec, _ = precision_recall_curve(y_test, knn_probs)
rf_prec, rf_rec, _ = precision_recall_curve(y_test, rf_probs)
xgb_prec, xgb_rec, _ = precision_recall_curve(y_test, xgb_probs)

plt.figure(figsize=(10, 6))
plt.plot(knn_rec, knn_prec, label=f'KNN (AUPRC = {auc(knn_rec, knn_prec):.3f})', color='green', lw=2)
plt.plot(rf_rec, rf_prec, label=f'Random Forest (AUPRC = {auc(rf_rec, rf_prec):.3f})', color='darkorange', lw=2)
plt.plot(xgb_rec, xgb_prec, label=f'XGBoost (AUPRC = {auc(xgb_rec, xgb_prec):.3f})', color='blue', lw=2)

plt.xlabel('Recall (True Positive Rate)')
plt.ylabel('Precision (Positive Predictive Value)')
plt.title('Precision-Recall Curve Comparison (Highly Imbalanced Data)')
plt.legend(loc='lower left')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 6. Recurrent Neural Network (RNN / LSTM)
# To process "Paying Styles", RNNs require 3D sequences: [samples, time_steps, features]
SEQUENCE_LENGTH = 5

def create_sequences(data, labels, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(labels[i + seq_length])
    return np.array(xs), np.array(ys)

X_sub = StandardScaler().fit_transform(X_train.iloc[:5000].values)
y_sub = y_train.iloc[:5000].values

X_seq, y_seq = create_sequences(X_sub, y_sub, SEQUENCE_LENGTH)
print(f"RNN/LSTM 3D Input Shape: {X_seq.shape}\n")

# Define the RNN (Using LSTM architecture for better long-term sequence memory)
rnn_model = Sequential([
    LSTM(32, input_shape=(X_seq.shape[1], X_seq.shape[2]), return_sequences=False),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(optimizer='adam', 
                   loss='binary_crossentropy', 
                   metrics=[tf.keras.metrics.AUC(curve='PR', name='auprc')])

print("Training RNN (LSTM) on sequential behavior...")
rnn_model.fit(X_seq, y_seq, epochs=3, batch_size=64, validation_split=0.2, verbose=1)